In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = Path(
    "/Users/jasminetaha/Documents/Retroverse Analytics"
)

CLEANED_DATA = BASE_PATH / "data" / "cleaned"

sales_orders = pd.read_csv(
    CLEANED_DATA / "sales_orders.csv"
)

sales_orders["Created at"] = pd.to_datetime(
    sales_orders["Created at"],
    errors="coerce",
    utc=True
)

print("Orders:", sales_orders["Id"].nunique())
print("Revenue:", sales_orders["Net Revenue"].sum())

Orders: 118
Revenue: 192606.4


In [2]:
sales_orders["Month"] = (
    sales_orders["Created at"]
    .dt.tz_convert(None)
    .dt.to_period("M")
)

monthly_sales = (
    sales_orders
    .groupby("Month")
    .agg(
        Orders=("Id", "nunique"),
        Revenue=("Net Revenue", "sum")
    )
)

all_months = pd.period_range(
    sales_orders["Month"].min(),
    sales_orders["Month"].max(),
    freq="M"
)

monthly_sales = (
    monthly_sales
    .reindex(all_months, fill_value=0)
    .rename_axis("Month")
    .reset_index()
)

monthly_sales

,Month,Orders,Revenue
0,2023-07,2,1449.0
1,2023-08,3,2600.0
2,2023-09,1,915.0
3,2023-10,1,965.0
4,2023-11,28,38203.0
5,2023-12,23,31300.0
6,2024-01,5,7200.0
7,2024-02,8,12750.0
8,2024-03,7,12400.0
9,2024-04,6,9800.0


In [3]:
sales_without_outlier = sales_orders[
    sales_orders["Net Revenue"] < 19000
].copy()

monthly_without_outlier = (
    sales_without_outlier
    .groupby("Month")
    .agg(
        Orders=("Id", "nunique"),
        Revenue=("Net Revenue", "sum")
    )
    .reindex(all_months, fill_value=0)
    .rename_axis("Month")
    .reset_index()
)

In [4]:
comparison = pd.DataFrame({
    "Month": monthly_sales["Month"].astype(str),
    "Actual_Revenue": monthly_sales["Revenue"],
    "Revenue_Without_19K": monthly_without_outlier["Revenue"],
    "Orders": monthly_sales["Orders"]
})

comparison.tail(18)

,Month,Actual_Revenue,Revenue_Without_19K,Orders
20,2025-03,700.0,700.0,1
21,2025-04,1800.0,1800.0,1
22,2025-05,1980.0,1980.0,2
23,2025-06,880.0,880.0,1
24,2025-07,1830.0,1830.0,2
25,2025-08,0.0,0.0,0
26,2025-09,0.0,0.0,0
27,2025-10,0.0,0.0,0
28,2025-11,0.0,0.0,0
29,2025-12,0.0,0.0,0


In [5]:
# Last 6 months
last_6 = monthly_without_outlier.tail(6)

# Last 12 months
last_12 = monthly_without_outlier.tail(12)

baseline_comparison = pd.DataFrame({
    "Period": [
        "Last 6 months",
        "Last 12 months"
    ],
    "Avg_Monthly_Revenue": [
        last_6["Revenue"].mean(),
        last_12["Revenue"].mean()
    ],
    "Avg_Monthly_Orders": [
        monthly_sales.tail(6)["Orders"].mean(),
        monthly_sales.tail(12)["Orders"].mean()
    ],
    "Active_Months": [
        (last_6["Revenue"] > 0).sum(),
        (last_12["Revenue"] > 0).sum()
    ]
})

baseline_comparison.round(2)

,Period,Avg_Monthly_Revenue,Avg_Monthly_Orders,Active_Months
0,Last 6 months,1208.33,0.67,2
1,Last 12 months,604.17,0.33,2


for the sales forecast im using a baseline scenario prediction bases on normal recent sales activate rather than all historical sales since they're unstable

In [6]:
active_recent = monthly_without_outlier[
    (monthly_without_outlier["Month"] >= pd.Period("2025-01", freq="M"))
    & (monthly_without_outlier["Revenue"] > 0)
]

print(
    "Average revenue in active months:",
    round(active_recent["Revenue"].mean(), 2)
)

print(
    "Average orders in active months:",
    round(
        monthly_sales[
            (monthly_sales["Month"] >= pd.Period("2025-01", freq="M"))
            & (monthly_sales["Orders"] > 0)
        ]["Orders"].mean(),
        2
    )
)

print(
    "Number of active months:",
    len(active_recent)
)

Average revenue in active months: 2957.78
Average orders in active months: 2.0
Number of active months: 9


In [7]:
baseline_scenarios = pd.DataFrame({
    "Scenario": [
        "Conservative",
        "Base",
        "Recovery"
    ],
    "Monthly_Revenue": [
        604.17,      # last 12-month average
        1208.33,     # last 6-month average
        2957.78      # average active month
    ]
})

baseline_scenarios["3_Month_Revenue"] = (
    baseline_scenarios["Monthly_Revenue"] * 3
)

baseline_scenarios["6_Month_Revenue"] = (
    baseline_scenarios["Monthly_Revenue"] * 6
)

baseline_scenarios["12_Month_Revenue"] = (
    baseline_scenarios["Monthly_Revenue"] * 12
)

baseline_scenarios.round(2)

,Scenario,Monthly_Revenue,3_Month_Revenue,6_Month_Revenue,12_Month_Revenue
0,Conservative,604.17,1812.51,3625.02,7250.04
1,Base,1208.33,3624.99,7249.98,14499.96
2,Recovery,2957.78,8873.34,17746.68,35493.36


### Baseline Forecast

Recent sales are very inconsistent, with several months having no orders.

Without the 19,000 EGP outlier, the last 6 months averaged around 1,208 EGP in monthly revenue. Active recent months averaged around 2,958 EGP.

Because the sales history is small and irregular, I used conservative, base and recovery scenarios instead of relying on one forecast.

Conservative: “What if sales stay weak?”
Base: “What if current performance continues?”
Recovery: “What if Retroverse starts consistently selling again?”

In [10]:
prediction_check = pd.DataFrame({
    "Metric": [
        "Total identifiable customers",
        "Repeat customers",
        "One-time customers",
        "Repeat rate"
    ],
    "Value": [
        len(rfm),
        (rfm["Frequency"] > 1).sum(),
        (rfm["Frequency"] == 1).sum(),
        (rfm["Frequency"] > 1).mean() * 100
    ]
})

prediction_check

,Metric,Value
0,Total identifiable customers,93.000000
1,Repeat customers,6.000000
2,One-time customers,87.000000
3,Repeat rate,6.451613


In [9]:
rfm = pd.read_csv(
    CLEANED_DATA / "customer_rfm_segments.csv"
)

print(rfm.shape)
rfm.head()

(93, 9)


,Customer_Key,Orders,Total_Spent,First_Order,Last_Order,Recency,Frequency,Monetary,Segment
0,email_abdullahrafatt@gmail.com,1,1600.0,2024-04-15 17:29:08+00:00,2024-04-15 17:29:08+00:00,853,1,1600.0,Lost / Inactive
1,email_ahmadkhalilll2021@gmail.com,1,850.0,2025-02-13 16:36:14+00:00,2025-02-13 16:36:14+00:00,549,1,850.0,Lost / Inactive
2,email_ahmed.mitry@nyu.edu,1,19000.0,2026-08-16 15:49:15+00:00,2026-08-16 15:49:15+00:00,1,1,19000.0,New / Recent
3,email_ahmeddssameh55@gmail.com,1,2130.0,2025-02-28 16:26:04+00:00,2025-02-28 16:26:04+00:00,534,1,2130.0,High Value - Inactive
4,email_ahmedelhakim07@gmail.com,1,1700.0,2024-04-15 20:11:15+00:00,2024-04-15 20:11:15+00:00,853,1,1700.0,Lost / Inactive


In [11]:
propensity = rfm.copy()

# Recency score: more recent customers score higher
propensity["Recency_Score"] = pd.cut(
    propensity["Recency"],
    bins=[-1, 180, 365, 730, np.inf],
    labels=[4, 3, 2, 1]
).astype(int)

# Frequency score: repeat buyers score higher
propensity["Frequency_Score"] = np.where(
    propensity["Frequency"] >= 2,
    3,
    1
)

# Monetary score: higher-spending customers score higher
propensity["Monetary_Score"] = pd.cut(
    propensity["Monetary"],
    bins=[-1, 1200, 1800, 3000, np.inf],
    labels=[1, 2, 3, 4]
).astype(int)

propensity["Propensity_Score"] = (
    propensity["Recency_Score"] * 0.4
    + propensity["Frequency_Score"] * 0.3
    + propensity["Monetary_Score"] * 0.3
)

propensity = propensity.sort_values(
    "Propensity_Score",
    ascending=False
)

propensity[
    [
        "Customer_Key",
        "Recency",
        "Frequency",
        "Monetary",
        "Segment",
        "Propensity_Score"
    ]
].head(15)

,Customer_Key,Recency,Frequency,Monetary,Segment,Propensity_Score
27,email_kikos96@hotmail.com,121,2,3770.0,Loyal / Active,3.7
2,email_ahmed.mitry@nyu.edu,1,1,19000.0,New / Recent,3.1
55,email_seleemabaza0@gmail.com,436,2,4530.0,Repeat - At Risk,2.9
5,email_ahmedika210@gmail.com,132,1,2490.0,New / Recent,2.8
20,email_janaessam@aucegypt.edu,2,1,2490.0,New / Recent,2.8
21,email_jojomahmoud602@gmail.com,411,2,1830.0,Repeat - At Risk,2.6
63,email_zamina_zidan@outlook.com,882,2,3300.0,Repeat - At Risk,2.5
22,email_josten.chris@icloud.com,707,1,3100.0,High Value - Inactive,2.3
23,email_kamaleldinaboulkheir@gmail.com,834,2,2950.0,Repeat - At Risk,2.2
66,email_ziadreda578@gmail.com,967,2,2500.0,Repeat - At Risk,2.2


### Repeat Purchase Prediction

Only 6 out of 93 customers purchased more than once, so there is not enough data to build a reliable machine learning model.

Instead, I ranked customers based on how recently they ordered, how many times they ordered, and how much they spent.

The goal is to identify which previous customers are more worth targeting to bring back.

In [12]:
propensity["Priority"] = pd.cut(
    propensity["Propensity_Score"],
    bins=[0, 2, 3, np.inf],
    labels=["Low", "Medium", "High"]
)

priority_summary = (
    propensity
    .groupby("Priority", observed=True)
    .agg(
        Customers=("Customer_Key", "count"),
        Average_Spend=("Monetary", "mean"),
        Total_Revenue=("Monetary", "sum")
    )
    .reset_index()
)

priority_summary["Customer_Share_%"] = (
    priority_summary["Customers"] / len(propensity) * 100
)

priority_summary.round(2)

,Priority,Customers,Average_Spend,Total_Revenue,Customer_Share_%
0,Low,83,1532.75,127218.4,89.25
1,Medium,8,2898.75,23190.0,8.60
2,High,2,11385.00,22770.0,2.15


### Customer Targeting

Most previous customers are low priority because they have not ordered in a long time.

10 customers stand out as better targets for reactivation, with 2 high-priority and 8 medium-priority customers.

These customers should be targeted first instead of spending money trying to reach every previous customer.

In [13]:
propensity.to_csv(
    CLEANED_DATA / "customer_propensity.csv",
    index=False
)

priority_summary.to_csv(
    CLEANED_DATA / "customer_priority_summary.csv",
    index=False
)

### Model Limitation

There are only 6 repeat customers in the dataset, which is not enough to train a reliable repeat-purchase machine learning model.

A customer ranking was used instead so the results stay based on the available data without making the prediction look more accurate than it really is.